In [1]:
import pyarrow.parquet as pq
import polars as pl
pl.Config.set_tbl_cols(40)          # Show up to 20 columns
pl.Config.set_tbl_rows(80)   
pl.Config.set_tbl_width_chars(500)  # Make the table wider in the console
pl.Config.set_fmt_str_lengths(50)   # Don't cut off long strings like stoch_key

results_path = r'C:\Users\Owner\airflow-trading\data_lake\Opt_Session_20260427_110253_01\master_metrics.parquet'

# Open the file metadata
parquet_file = pq.ParquetFile(results_path)

df_master = pl.read_parquet(results_path)

# 1. Get Row Count
print(f"Total Rows: {parquet_file.metadata.num_rows}")

# 2. Get Schema
print("Master Data Schema:")
print(parquet_file.schema)

Total Rows: 12600
Master Data Schema:
required group field_id=-1 schema {
  optional int32 field_id=-1 regime_id;
  optional int32 field_id=-1 signal_layer (Int(bitWidth=8, isSigned=true));
  optional binary field_id=-1 signal_scope_id (String);
  optional int64 field_id=-1 era_int;
  optional int32 field_id=-1 exit_window_h;
  optional float field_id=-1 SL;
  optional float field_id=-1 TP;
  optional boolean field_id=-1 use_trailing_sl;
  optional float field_id=-1 trailing_sl_pct;
  optional int32 field_id=-1 trailing_sl_interval;
  optional boolean field_id=-1 trailing_sl_stop_at_pos;
  optional boolean field_id=-1 use_limit_entry;
  optional int32 field_id=-1 limit_order_expiry_bars;
  optional boolean field_id=-1 trade_overlap;
  optional boolean field_id=-1 trade_flip_on_entry;
  optional int32 field_id=-1 trade_window_interval;
  optional int32 field_id=-1 total_pos;
  optional int32 field_id=-1 win_pos;
  optional float field_id=-1 balance;
  optional float field_id=-1 max_draw

In [7]:
# Aggregate with Drawdown metrics and strict consistency filter
robust_combos_with_risk = (
    df_master
    .filter(pl.col("trade_overlap") == True)
    .group_by([
        "regime_id", "signal_layer", "signal_scope_id", "exit_window_h", 
        "SL", "TP", "use_trailing_sl", "trade_flip_on_entry"
    ])
    .agg(
        pl.col("balance").mean().alias("avg_balance"),
        pl.col("balance").min().alias("min_era_balance"),
        pl.col("max_drawdown").max().alias("worst_drawdown"), # The deepest dip ever recorded
        pl.col("max_drawdown").mean().alias("avg_drawdown"),
        pl.col("era_int").n_unique().alias("total_eras"),
        pl.col("total_pos").sum().alias("total_trades")
    )
    .filter(pl.col("min_era_balance") >= 100)
    .sort("avg_balance", descending=True)
)

print("Robust Combos (Min Balance > ) ranked by Avg Balance with Drawdown:")
print(robust_combos_with_risk)

Robust Combos (Min Balance > ) ranked by Avg Balance with Drawdown:
shape: (8, 14)
┌───────────┬──────────────┬──────────────────┬───────────────┬─────┬─────┬─────────────────┬─────────────────────┬─────────────┬─────────────────┬────────────────┬──────────────┬────────────┬──────────────┐
│ regime_id ┆ signal_layer ┆ signal_scope_id  ┆ exit_window_h ┆ SL  ┆ TP  ┆ use_trailing_sl ┆ trade_flip_on_entry ┆ avg_balance ┆ min_era_balance ┆ worst_drawdown ┆ avg_drawdown ┆ total_eras ┆ total_trades │
│ ---       ┆ ---          ┆ ---              ┆ ---           ┆ --- ┆ --- ┆ ---             ┆ ---                 ┆ ---         ┆ ---             ┆ ---            ┆ ---          ┆ ---        ┆ ---          │
│ i32       ┆ i8           ┆ str              ┆ i32           ┆ f32 ┆ f32 ┆ bool            ┆ bool                ┆ f32         ┆ f32             ┆ f32            ┆ f32          ┆ u32        ┆ i32          │
╞═══════════╪══════════════╪══════════════════╪═══════════════╪═════╪═════╪══════════

In [5]:
# --- Configuration ---
TARGET_REGIME_ID = 92

# Ensure we can see the full string for signal_scope_id
pl.Config.set_fmt_str_lengths(200) 

# 1. Filter for the specific regime and aggregate
regime_summary = (
    df_master
    .filter(pl.col("regime_id") == TARGET_REGIME_ID)
    .group_by("regime_id")
    .agg([
        # Parameters (taking first since they are identical across eras for one regime)
        pl.col("signal_scope_id").first(),
        pl.col("signal_layer").first(),
        pl.col("exit_window_h").first(),
        pl.col("SL").first(),
        pl.col("TP").first(),
        pl.col("use_trailing_sl").first(),
        pl.col("trade_overlap").first(),
        pl.col("trade_flip_on_entry").first(),
        
        # Results (Aggregating across all eras)
        pl.col("era_int").n_unique().alias("total_eras_sampled"),
        pl.col("balance").mean().alias("avg_balance"),
        pl.col("balance").min().alias("worst_era_balance"),
        pl.col("max_drawdown").max().alias("max_drawdown_peak"),
        pl.col("total_pos").sum().alias("total_trades_all_eras"),
        
        # The full JSON if you need to inspect it
        pl.col("signal_json").first()
    ])
)

# 2. Print full signal_scope_id separately for clarity
if not regime_summary.is_empty():
    full_scope_id = regime_summary["signal_scope_id"]
    print(f"--- Full Analysis for Regime ID: {TARGET_REGIME_ID} ---")
    print(f"Full signal_scope_id: {full_scope_id}")
    print("-" * 50)
    print(regime_summary)
else:
    print(f"Regime ID {TARGET_REGIME_ID} not found in the dataset.")

--- Full Analysis for Regime ID: 92 ---
Full signal_scope_id: shape: (1,)
Series: 'signal_scope_id' [str]
[
	"stoch__15m__k12_d8_s4_l30_u70_tol10|lookback__5m__u2"
]
--------------------------------------------------
shape: (1, 15)
┌───────────┬──────────────────────────────────────────────────────┬──────────────┬───────────────┬─────┬─────┬─────────────────┬───────────────┬─────────────────────┬────────────────────┬─────────────┬───────────────────┬───────────────────┬───────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ regime_id ┆ signal_scope_id                                      ┆ signal_layer ┆ exit_window_h ┆ SL  ┆ TP  ┆ use_trailing_sl ┆ trade_overlap ┆ trade_flip_on_entry ┆ total_eras_sampled ┆ avg_balance ┆ worst_era_balance ┆ max_drawdown_peak ┆ total_trades_all_eras ┆ signal_json                   

In [4]:
# --- Baseline verification cell ---

# Heuristic definition of "baseline":
# 1) empty signal_scope_id OR
# 2) signal_scope_id contains 'baseline'
baseline_df = df_master.filter(
    (pl.col("signal_scope_id").is_null()) |
    (pl.col("signal_scope_id") == "") |
    (pl.col("signal_scope_id").str.contains("baseline", literal=False))
)

print("Baseline rows:", baseline_df.height)

if baseline_df.height == 0:
    print("❌ No baseline signal found at all")
else:
    print("✅ Baseline signal exists")

    # Basic sanity stats
    baseline_summary = (
        baseline_df
        .select([
            pl.col("era_int").n_unique().alias("eras"),
            pl.col("regime_id").n_unique().alias("regimes"),
            pl.col("balance").min().alias("min_balance"),
            pl.col("balance").max().alias("max_balance"),
            pl.col("balance").mean().alias("mean_balance"),
        ])
    )
    print("\nBaseline summary:")
    display(baseline_summary)

    # Check if baseline ever beats starting equity
    profitable_baseline = baseline_df.filter(pl.col("balance") > 100.0)

    print("\nBaseline rows with balance > 100:", profitable_baseline.height)

    if profitable_baseline.height == 0:
        print("❌ Baseline never exceeds 100 balance")
    else:
        print("✅ Baseline produces equity growth")

        # Show best baseline rows
        display(
            profitable_baseline
            .sort("balance", descending=True)
            .select([
                "era_int",
                "regime_id",
                "signal_scope_id",
                "SL",
                "TP",
                "total_pos",
                "win_pos",
                "balance",
                "max_drawdown",
            ])
            .head(20)
        )

Baseline rows: 3600
✅ Baseline signal exists

Baseline summary:


eras,regimes,min_balance,max_balance,mean_balance
u32,u32,f32,f32,f32
12,300,89.468056,123.09539,100.327591



Baseline rows with balance > 100: 1832
✅ Baseline produces equity growth


era_int,regime_id,signal_scope_id,SL,TP,total_pos,win_pos,balance,max_drawdown
i64,i32,str,f32,f32,i32,i32,f32,f32
20241201,561,"""baseline:all_buy""",0.6,0.8,92,55,123.09539,0.023959
20241201,563,"""baseline:all_sell""",0.6,0.8,92,55,123.09539,0.023959
20241201,813,"""baseline:all_buy""",0.8,1.0,82,48,121.893219,0.047847
20241201,815,"""baseline:all_sell""",0.8,1.0,82,48,121.893219,0.047847
20241201,910,"""baseline:all_buy""",1.0,0.4,93,78,120.495064,0.023152
20241201,912,"""baseline:all_sell""",1.0,0.4,93,78,120.495064,0.023152
20241201,981,"""baseline:all_buy""",1.0,0.8,85,58,120.230873,0.041408
20241201,983,"""baseline:all_sell""",1.0,0.8,85,58,120.230873,0.041408
20231201,1023,"""baseline:all_buy""",1.0,1.0,82,50,119.780968,0.063808


In [10]:
from pathlib import Path
import polars as pl

session_dir = Path(r"C:\Users\Owner\airflow-trading\data_lake\Opt_Session_20260426_092617_01")
regime_id = 92

master_path = session_dir / "master_metrics.parquet"
df_master = pl.read_parquet(master_path)

trade_ml_root = session_dir / "trade_ml_partitioned"
trade_paths = sorted(trade_ml_root.glob("era_int=*/trade_ml_era_int=*.parquet"))

df_trade = pl.concat([pl.read_parquet(str(p)) for p in trade_paths], how="vertical") if trade_paths else pl.DataFrame()

print("MASTER rows for regime", regime_id)
print(
    df_master
    .filter(pl.col("regime_id") == regime_id)
    .select(["regime_id", "signal_layer", "signal_scope_id", "era_int", "SL", "TP", "total_pos", "win_pos", "balance", "max_drawdown", "max_consecutive_losses"])
    .sort(["era_int", "SL", "TP"])
)

print("\nTRADE_ML rows for regime", regime_id)
if df_trade.is_empty():
    print("trade_ml is empty")
else:
    print(
        df_trade
        .filter(pl.col("regime_id") == regime_id)
        .select(["regime_id", "signal_layer", "signal_scope_id", "era_int", "SL", "TP", "signal_idx", "entry_idx", "exit_idx", "pnl_pct"])
        .sort(["era_int", "SL", "TP"])
    )

print("\nDoes regime exist in trade_ml?", not df_trade.filter(pl.col("regime_id") == regime_id).is_empty())

MASTER rows for regime 92
shape: (300, 11)
┌───────────┬──────────────┬──────────────────────────────────────────────────────┬──────────┬─────┬─────┬───────────┬─────────┬────────────┬──────────────┬────────────────────────┐
│ regime_id ┆ signal_layer ┆ signal_scope_id                                      ┆ era_int  ┆ SL  ┆ TP  ┆ total_pos ┆ win_pos ┆ balance    ┆ max_drawdown ┆ max_consecutive_losses │
│ ---       ┆ ---          ┆ ---                                                  ┆ ---      ┆ --- ┆ --- ┆ ---       ┆ ---     ┆ ---        ┆ ---          ┆ ---                    │
│ i32       ┆ i8           ┆ str                                                  ┆ i64      ┆ f32 ┆ f32 ┆ i32       ┆ i32     ┆ f32        ┆ f32          ┆ i32                    │
╞═══════════╪══════════════╪══════════════════════════════════════════════════════╪══════════╪═════╪═════╪═══════════╪═════════╪════════════╪══════════════╪════════════════════════╡
│ 92        ┆ 2            ┆ stoch__15m__k12_d8

In [11]:
from pathlib import Path
import json
import polars as pl

session_dir = Path(r"C:\Users\Owner\airflow-trading\data_lake\Opt_Session_20260426_092617_01")
cfg_dir = session_dir / "configs"

rows = []
for p in sorted(cfg_dir.glob("batch_*.json")):
    with open(p, "r", encoding="utf8") as f:
        batch = json.load(f)
    for r in batch.get("regimes", []):
        if int(r.get("regime_id", -1)) == 92:
            rows.append({
                "batch": p.name,
                "regime_id": r.get("regime_id"),
                "regime_key": r.get("regime_key"),
                "signal_layer": r.get("signal_layer"),
                "signal_scope_id": r.get("signal_scope_id"),
                "SL": r.get("SL"),
                "TP": r.get("TP"),
            })

df = pl.DataFrame(rows) if rows else pl.DataFrame(schema={
    "batch": pl.Utf8,
    "regime_id": pl.Int64,
    "regime_key": pl.Utf8,
    "signal_layer": pl.Int64,
    "signal_scope_id": pl.Utf8,
    "SL": pl.Float64,
    "TP": pl.Float64,
})

print("Current config entries for regime_id 92:")
print(df)

if df.is_empty():
    print("\nRegime 92 is not present in the current configs, so the master row is stale/mixed from an older run.")

Current config entries for regime_id 92:
shape: (1, 7)
┌─────────────────┬───────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────┬──────────────────────────────────────────────────────┬─────┬─────┐
│ batch           ┆ regime_id ┆ regime_key                                                                                                                                                                                                ┆ signal_layer ┆ signal_scope_id                                      ┆ SL  ┆ TP  │
│ ---             ┆ ---       ┆ ---                                                                                                                                                                                                       ┆ ---          ┆ ---                                                  ┆ ---

In [12]:
from pathlib import Path
import json
import polars as pl

session_dir = Path(r"C:\Users\Owner\airflow-trading\data_lake\Opt_Session_20260426_092617_01")
cfg_dir = session_dir / "configs"

rows = []
for p in sorted(cfg_dir.glob("batch_*.json")):
    with open(p, "r", encoding="utf8") as f:
        batch = json.load(f)
    for r in batch.get("regimes", []):
        rows.append({
            "batch": p.name,
            "regime_id": r.get("regime_id"),
            "regime_key": r.get("regime_key"),
            "signal_layer": r.get("signal_layer"),
            "signal_scope_id": r.get("signal_scope_id"),
        })

df_cfg = pl.DataFrame(rows)

dup_keys = (
    df_cfg
    .group_by("regime_key")
    .agg(
        pl.col("regime_id").n_unique().alias("regime_id_count"),
        pl.col("regime_id").unique().alias("regime_ids"),
        pl.col("batch").n_unique().alias("batch_count"),
        pl.col("batch").unique().alias("batches"),
        pl.col("signal_layer").first().alias("signal_layer"),
        pl.col("signal_scope_id").first().alias("signal_scope_id"),
    )
    .filter(pl.col("regime_id_count") > 1)
    .sort("regime_id_count", descending=True)
)

print("Duplicate regime_key mappings:", dup_keys.height)
print(dup_keys.head(20))

Duplicate regime_key mappings: 0
shape: (0, 7)
┌────────────┬─────────────────┬────────────┬─────────────┬───────────┬──────────────┬─────────────────┐
│ regime_key ┆ regime_id_count ┆ regime_ids ┆ batch_count ┆ batches   ┆ signal_layer ┆ signal_scope_id │
│ ---        ┆ ---             ┆ ---        ┆ ---         ┆ ---       ┆ ---          ┆ ---             │
│ str        ┆ u32             ┆ list[i64]  ┆ u32         ┆ list[str] ┆ i64          ┆ str             │
╞════════════╪═════════════════╪════════════╪═════════════╪═══════════╪══════════════╪═════════════════╡
└────────────┴─────────────────┴────────────┴─────────────┴───────────┴──────────────┴─────────────────┘


In [2]:
# Map era -> volatility

amp_data = [
    {
        "era_int": 20220901,
        "window_start_time": "2022-09-01 00:00:00+00:00",
        "window_end_time": "2022-12-01 00:00:00+00:00",
        "timeframe_months": 3,
        "window_rows": 26208,
        "window_min_close": 15593.58,
        "window_max_close": 22713.69,
        "regime_amp_index": 0.456605
    },
    {
        "era_int": 20221201,
        "window_start_time": "2022-12-01 00:00:00+00:00",
        "window_end_time": "2023-03-01 00:00:00+00:00",
        "timeframe_months": 3,
        "window_rows": 25920,
        "window_min_close": 16366.60,
        "window_max_close": 25235.36,
        "regime_amp_index": 0.541882
    },
    {
        "era_int": 20230301,
        "window_start_time": "2023-03-01 00:00:00+00:00",
        "window_end_time": "2023-06-01 00:00:00+00:00",
        "timeframe_months": 3,
        "window_rows": 26496,
        "window_min_close": 19611.25,
        "window_max_close": 30963.13,
        "regime_amp_index": 0.578845
    },
    {
        "era_int": 20230601,
        "window_start_time": "2023-06-01 00:00:00+00:00",
        "window_end_time": "2023-09-01 00:00:00+00:00",
        "timeframe_months": 3,
        "window_rows": 26496,
        "window_min_close": 24891.49,
        "window_max_close": 31710.95,
        "regime_amp_index": 0.273968
    },
    {
        "era_int": 20230901,
        "window_start_time": "2023-09-01 00:00:00+00:00",
        "window_end_time": "2023-12-01 00:00:00+00:00",
        "timeframe_months": 3,
        "window_rows": 26208,
        "window_min_close": 25003.56,
        "window_max_close": 38383.09,
        "regime_amp_index": 0.535105
    },
    {
        "era_int": 20231201,
        "window_start_time": "2023-12-01 00:00:00+00:00",
        "window_end_time": "2024-03-01 00:00:00+00:00",
        "timeframe_months": 3,
        "window_rows": 26208,
        "window_min_close": 37627.91,
        "window_max_close": 63690.12,
        "regime_amp_index": 0.692630
    },
    {
        "era_int": 20240301,
        "window_start_time": "2024-03-01 00:00:00+00:00",
        "window_end_time": "2024-06-01 00:00:00+00:00",
        "timeframe_months": 3,
        "window_rows": 26496,
        "window_min_close": 56773.99,
        "window_max_close": 73628.42,
        "regime_amp_index": 0.296869
    },
    {
        "era_int": 20240601,
        "window_start_time": "2024-06-01 00:00:00+00:00",
        "window_end_time": "2024-09-01 00:00:00+00:00",
        "timeframe_months": 3,
        "window_rows": 26496,
        "window_min_close": 49338.49,
        "window_max_close": 71956.73,
        "regime_amp_index": 0.458430
    },
    {
        "era_int": 20240901,
        "window_start_time": "2024-09-01 00:00:00+00:00",
        "window_end_time": "2024-12-01 00:00:00+00:00",
        "timeframe_months": 3,
        "window_rows": 26208,
        "window_min_close": 52738.00,
        "window_max_close": 99521.87,
        "regime_amp_index": 0.887100
    },
    {
        "era_int": 20241201,
        "window_start_time": "2024-12-01 00:00:00+00:00",
        "window_end_time": "2025-03-01 00:00:00+00:00",
        "timeframe_months": 3,
        "window_rows": 25920,
        "window_min_close": 78487.01,
        "window_max_close": 108984.96,
        "regime_amp_index": 0.388573
    },
    {
        "era_int": 20250301,
        "window_start_time": "2025-03-01 00:00:00+00:00",
        "window_end_time": "2025-06-01 00:00:00+00:00",
        "timeframe_months": 3,
        "window_rows": 26496,
        "window_min_close": 74610.00,
        "window_max_close": 111924.08,
        "regime_amp_index": 0.500122
    },
    {
        "era_int": 20250601,
        "window_start_time": "2025-06-01 00:00:00+00:00",
        "window_end_time": "2025-09-01 00:00:00+00:00",
        "timeframe_months": 3,
        "window_rows": 26496,
        "window_min_close": 98392.91,
        "window_max_close": 124243.32,
        "regime_amp_index": 0.262726
    },
    {
        "era_int": 20250901,
        "window_start_time": "2025-09-01 00:00:00+00:00",
        "window_end_time": "2025-09-15 23:59:59+00:00",
        "timeframe_months": 3,
        "window_rows": 4320,
        "window_min_close": 107303.99,
        "window_max_close": 116745.67,
        "regime_amp_index": 0.087990
    }
]
import polars as pl

# 1. Convert your list of dictionaries into a Polars DataFrame
# We only select 'era_int' and 'regime_amp_index' as those are the keys for correlation
amp_df = (
    pl.DataFrame(amp_data)
    .select([
        pl.col("era_int"),
        pl.col("regime_amp_index").alias("regime_amp")  # Rename to match your analysis script
    ])
    .with_columns(pl.col("era_int").cast(pl.Int64)) # Ensure data types match for the join
)

# 2. Link the Volatility data to your Master Backtest data
# This is the 'bridge' that allows you to calculate correlation
df_analysis = (
    df_master
    .with_columns(pl.col("era_int").cast(pl.Int64)) # Ensure join key types are identical
    .join(amp_df, on="era_int", how="left")        # Map volatility to every single trade result
    .filter(pl.col("total_pos") >= 30)             # Apply trade floor for significance
    .with_columns([
        # Volatility-Adjusted Alpha Calculation
        ((((pl.col("balance") - 100) / 100) / (pl.col("total_pos") + 1e-9)) / 
         (pl.col("max_drawdown").clip(0.001))).alias("alpha_per_trade"),
        
        # Total Trade Space (TP + SL)
        (pl.col("TP") + pl.col("SL")).alias("total_spread")
    ])
)

# 3. Aggregate to find the "Best" parameters for every Volatility level
regime_winners = (
    df_analysis
    .group_by(["regime_amp", "TP", "SL", "total_spread"])
    .agg(pl.col("alpha_per_trade").median().alias("median_alpha"))
    .sort(["regime_amp", "median_alpha"], descending=[False, True])
    .group_by("regime_amp")
    .head(1)
    .sort("regime_amp")
)

# 4. Final Spearman Rho Check
correlations = regime_winners.select([
    pl.corr("regime_amp", "TP", method="spearman").alias("rho_vol_vs_TP"),
    pl.corr("regime_amp", "SL", method="spearman").alias("rho_vol_vs_SL"),
    pl.corr("regime_amp", "total_spread", method="spearman").alias("rho_vol_vs_TotalSpread")
])

print("--- IDEAL PARAMETERS PER VOLATILITY REGIME ---")
print(regime_winners.select(["regime_amp", "TP", "SL", "total_spread", "median_alpha"]))

print("\n--- GLOBAL CORRELATION RESULTS ---")
print(correlations)

--- IDEAL PARAMETERS PER VOLATILITY REGIME ---
shape: (12, 5)
┌────────────┬─────┬─────┬──────────────┬──────────────┐
│ regime_amp ┆ TP  ┆ SL  ┆ total_spread ┆ median_alpha │
│ ---        ┆ --- ┆ --- ┆ ---          ┆ ---          │
│ f64        ┆ f32 ┆ f32 ┆ f32          ┆ f32          │
╞════════════╪═════╪═════╪══════════════╪══════════════╡
│ 0.262726   ┆ 4.5 ┆ 0.4 ┆ 4.9          ┆ -0.000641    │
│ 0.273968   ┆ 3.0 ┆ 0.2 ┆ 3.2          ┆ 0.002647     │
│ 0.296869   ┆ 6.0 ┆ 0.4 ┆ 6.4          ┆ 0.015605     │
│ 0.388573   ┆ 5.5 ┆ 0.2 ┆ 5.7          ┆ 0.022873     │
│ 0.456605   ┆ 4.0 ┆ 0.2 ┆ 4.2          ┆ 0.028315     │
│ 0.45843    ┆ 3.0 ┆ 0.2 ┆ 3.2          ┆ 0.066889     │
│ 0.500122   ┆ 2.0 ┆ 0.4 ┆ 2.4          ┆ 0.019333     │
│ 0.535105   ┆ 2.0 ┆ 0.2 ┆ 2.2          ┆ 0.00779      │
│ 0.541882   ┆ 2.0 ┆ 0.4 ┆ 2.4          ┆ -0.000527    │
│ 0.578845   ┆ 4.0 ┆ 0.4 ┆ 4.4          ┆ 0.00204      │
│ 0.69263    ┆ 4.5 ┆ 0.4 ┆ 4.9          ┆ 0.009275     │
│ 0.8871     ┆ 2.5 ┆ 0.4 ┆

In [ ]:
from feature_perf import feature_performance

results_stoch_15m = feature_performance(df_master, "stoch_key (15m)")
results_tp = feature_performance(df_master, "lookback_key (5m)")
results_tp = feature_performance(df_master, "lookback_key (15m)")

results_exit = feature_performance(df_master, "exit_window_h")
results_limit = feature_performance(df_master, "trailing_sl_pct")
results_limit_ex = feature_performance(df_master, "limit_order_expiry_bars")
results_interval = feature_performance(df_master, "trade_window_interval")



--- STOCH_KEY (15M) PERFORMANCE BY ERA (FLOOR: 40) ---
shape: (12, 8)
┌──────────┬───────────────────┬────────────────┬──────────┬──────────────┬───────────────┬─────────────────────────┬─────────────────┐
│ era_int  ┆ stoch_key (15m)   ┆ median_balance ┆ avg_dd   ┆ avg_win_rate ┆ avg_total_pos ┆ median_return_per_trade ┆ alpha_per_trade │
│ ---      ┆ ---               ┆ ---            ┆ ---      ┆ ---          ┆ ---           ┆ ---                     ┆ ---             │
│ i64      ┆ str               ┆ f32            ┆ f32      ┆ f64          ┆ f64           ┆ f64                     ┆ f64             │
╞══════════╪═══════════════════╪════════════════╪══════════╪══════════════╪═══════════════╪═════════════════════════╪═════════════════╡
│ 20220901 ┆ k12_d8_s4_l30_u70 ┆ 93.705788      ┆ 0.080797 ┆ 0.30434      ┆ 63.828103     ┆ -0.000986               ┆ -0.012193       │
│ 20221201 ┆ k12_d8_s4_l30_u70 ┆ 89.881866      ┆ 0.098546 ┆ 0.31577      ┆ 59.863515     ┆ -0.00169             

KeyError: "Feature column not found: lookback_key (1h)\nTried: ['lookback_key (1h)', 'signal__lookback_key (1h)']\nAvailable signal columns: ['signal__lookback__15m__entry_lookback_units', 'signal__lookback__15m__timeframe', 'signal__lookback__5m__entry_lookback_units', 'signal__lookback__5m__timeframe', 'signal__stochastic__15m__d', 'signal__stochastic__15m__k', 'signal__stochastic__15m__s', 'signal__stochastic__15m__threshold_tolerance', 'signal__stochastic__15m__thresholds', 'signal__stochastic__15m__timeframe', 'signal__stochastic__5m__d', 'signal__stochastic__5m__k', 'signal__stochastic__5m__s', 'signal__stochastic__5m__threshold_tolerance', 'signal__stochastic__5m__thresholds', 'signal__stochastic__5m__timeframe']\nAvailable human-readable columns: ['stoch_key (5m)', 'lookback_key (5m)', 'stoch_key (15m)', 'lookback_key (15m)']"

In [ ]:
# --- Top 10 Feature Combinations by Consistency then Alpha ---
def top10_combinations(df_master: pl.DataFrame, feature_cols: list, trade_floor: int = 50, top_pct: float = 0.01, min_return: float = 0.05):
    """
    Find top 10 combinations of features based on era consistency and alpha per trade.
    """
    # --- 0. Filter by trade floor ---
    df_filtered = (
        df_master
        .filter(pl.col("total_pos") >= trade_floor)
        .filter(((pl.col("balance") - 100) / 100) >= min_return)
    )

    if df_filtered.is_empty():
        print(
            f"No strategies found with total_pos >= {trade_floor} "
            f"and return >= {min_return:.0%}"
        )
        return None, None

    # --- 1. Compute alpha_per_trade for all rows ---
    df_filtered = df_filtered.with_columns([
        (((pl.col("balance") - 100) / 100) / (pl.col("total_pos") + 1e-9) / (pl.col("max_drawdown") + 1e-9)).alias("alpha_per_trade")
    ])

    # --- 2. Compute top elite per era ---
    eras = df_filtered["era_int"].unique().to_list()
    top_rows_per_era = []

    for era in eras:
        era_df = df_filtered.filter(pl.col("era_int") == era)
        cutoff = max(1, int(len(era_df) * top_pct))
        top_rows = era_df.sort("alpha_per_trade", descending=True).head(cutoff)
        top_rows_per_era.append(top_rows)

    top_elite_per_era = pl.concat(top_rows_per_era)

    # --- 3. Compute era consistency score for each combination ---
    era_counts = (
        top_elite_per_era
        .group_by(feature_cols)
        .agg(pl.col("era_int").n_unique().alias("eras_found"))
        .with_columns((pl.col("eras_found") / len(eras)).alias("era_consistency_score"))
    )

    # --- 4. Merge consistency back to full dataframe ---
    df_with_consistency = df_filtered.join(era_counts, on=feature_cols, how="left").fill_null(0.0)

    # --- 5. Compute top 10 combinations ---
    top10_combos = df_with_consistency.sort(
        ["era_consistency_score", "alpha_per_trade"],
        descending=[True, True]
    ).select(feature_cols + ["era_consistency_score", "alpha_per_trade"]).unique().head(10)

    # --- 6. Compute alpha lift vs global median ---
    global_median_alpha = df_with_consistency["alpha_per_trade"].median()
    top10_combos = top10_combos.with_columns([
        (pl.col("alpha_per_trade") - global_median_alpha).alias("alpha_lift")
    ])

    print(f"\n--- TOP 10 FEATURE COMBINATIONS ({', '.join(feature_cols)}) ---")
    print(top10_combos.sort(["era_consistency_score", "alpha_per_trade"], descending=True))

    return top10_combos

# Example usage: combinations of ma_int, entry_lookback_units, exit_window_h
top10_combo = top10_combinations(
    df_master, 
    feature_cols=["ma_int", "entry_lookback_units", "exit_window_h", "trade_window_interval", "sl_decay_pct", "SL", "TP"]
)


--- TOP 10 FEATURE COMBINATIONS (ma_int, entry_lookback_units, exit_window_h, trade_window_interval, trailing_sl_pct, SL, TP) ---
shape: (10, 10)
┌────────┬──────────────────────┬───────────────┬───────────────────────┬─────────────────┬─────┬─────┬───────────────────────┬─────────────────┬────────────┐
│ ma_int ┆ entry_lookback_units ┆ exit_window_h ┆ trade_window_interval ┆ trailing_sl_pct ┆ SL  ┆ TP  ┆ era_consistency_score ┆ alpha_per_trade ┆ alpha_lift │
│ ---    ┆ ---                  ┆ ---           ┆ ---                   ┆ ---             ┆ --- ┆ --- ┆ ---                   ┆ ---             ┆ ---        │
│ f64    ┆ f64                  ┆ f64           ┆ f64                   ┆ f32             ┆ f32 ┆ f32 ┆ f64                   ┆ f32             ┆ f32        │
╞════════╪══════════════════════╪═══════════════╪═══════════════════════╪═════════════════╪═════╪═════╪═══════════════════════╪═════════════════╪════════════╡
│ 2.0    ┆ 2.0                  ┆ 4.0           ┆ 24.0    

In [24]:
import polars as pl

def recursive_elite_selection(
    df_master: pl.DataFrame,
    feature_cols: list,
    top_pct: float = 0.1,
    trade_floor: int = 50
):
    """
    Recursive elite selection aligned with feature_performance logic:
    - Selection is PER ERA
    - Ranking metric: efficiency_per_trade
    - Feature value is chosen by:
        1) Era consistency
        2) Total top appearances (dominance)
        3) Efficiency per trade (tie-breaker only)
    """

    df_filtered = df_master.filter(pl.col("total_pos") >= trade_floor)
    elite_summary = []

    for feature in feature_cols:
        # --- 1. Compute efficiency_per_trade ---
        df_eff = df_filtered.with_columns([
            ((pl.col("balance") / (pl.col("max_drawdown") + 1e-9)) /
             (pl.col("total_pos") + 1e-9)).alias("efficiency_per_trade")
        ])

        eras = df_eff["era_int"].unique().to_list()
        top_rows_all_eras = []

        # --- 2. Select top X% PER ERA ---
        for era in eras:
            era_df = df_eff.filter(pl.col("era_int") == era)
            if era_df.is_empty():
                continue

            cutoff = max(1, int(era_df.height * top_pct))
            top_rows = (
                era_df
                .sort("efficiency_per_trade", descending=True)
                .head(cutoff)
            )
            top_rows_all_eras.append(top_rows)

        if not top_rows_all_eras:
            break

        top_elite = pl.concat(top_rows_all_eras)

        # --- 3. Era dominance scoring ---
        feature_score = (
            top_elite
            .group_by(feature)
            .agg([
                pl.col("era_int").n_unique().alias("eras_present"),
                pl.len().alias("total_top_instances"),
                pl.col("efficiency_per_trade").median().alias("median_eff_per_trade")
            ])
            .with_columns([
                (pl.col("eras_present") / len(eras)).alias("era_consistency_score")
            ])
            .sort(
                [
                    "era_consistency_score",   # must be consistent
                    "total_top_instances",     # dominance matters most
                    "median_eff_per_trade"     # quality as tie-breaker
                ],
                descending=[True, True, True]
            )
        )

        # --- 4. Pick dominant feature value ---
        top_value = feature_score[0, feature]

        elite_summary.append({
            "feature": feature,
            "top_value": top_value,
            "era_consistency": feature_score[0, "era_consistency_score"],
            "eras_present": feature_score[0, "eras_present"],
            "total_top_instances": feature_score[0, "total_top_instances"],
            "median_eff_per_trade": feature_score[0, "median_eff_per_trade"]
        })

        # --- 5. Filter for next recursion ---
        df_filtered = df_filtered.filter(pl.col(feature) == top_value)

        if df_filtered.height <= 1:
            break

    print("--- ELITE FEATURE COMBINATION SUMMARY (ERA DOMINANCE) ---")
    for e in elite_summary:
        print(e)

    print(f"\nFinal filtered dataset: {df_filtered.height} rows")
    return df_filtered, elite_summary

selected_features = [
    "stoch_key", "ma_int", "entry_lookback_units", 
    "exit_window_h", "TP", "SL", "sl_decay_pct", "use_sl_decay"
]

selected_features = [
    "trade_window_interval", "stoch_key", "ma_int", "entry_lookback_units", 
    "exit_window_h", "TP", "SL", "sl_decay_pct", "use_sl_decay"
]

df_elite, summary = recursive_elite_selection(df_master, selected_features)

--- ELITE FEATURE COMBINATION SUMMARY (ERA DOMINANCE) ---
{'feature': 'trade_window_interval', 'top_value': 12, 'era_consistency': 1.0, 'eras_present': 12, 'total_top_instances': 15736, 'median_eff_per_trade': 115.45074462890625}
{'feature': 'stoch_key', 'top_value': 'k6_d6_s6_l30_u70', 'era_consistency': 1.0, 'eras_present': 12, 'total_top_instances': 4704, 'median_eff_per_trade': 128.41375732421875}
{'feature': 'ma_int', 'top_value': 3, 'era_consistency': 1.0, 'eras_present': 12, 'total_top_instances': 1808, 'median_eff_per_trade': 129.90927124023438}
{'feature': 'entry_lookback_units', 'top_value': 2, 'era_consistency': 1.0, 'eras_present': 12, 'total_top_instances': 479, 'median_eff_per_trade': 148.0550537109375}
{'feature': 'exit_window_h', 'top_value': 1, 'era_consistency': 0.9166666666666666, 'eras_present': 11, 'total_top_instances': 389, 'median_eff_per_trade': 142.051513671875}
{'feature': 'TP', 'top_value': 2.200000047683716, 'era_consistency': 1.0, 'eras_present': 12, 'tota